In [153]:
import pandas as pd

In [154]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

!curl $data -o data.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  272k  100  272k    0     0   638k      0 --:--:-- --:--:-- --:--:--  637k


In [155]:
df = pd.read_csv('data.csv')
df.head().T

,0,1,2,3,4
lead_source,organic_search,social_media,referral,paid_ads,referral
industry,technology,technology,retail,manufacturing,manufacturing
employment_status,employed,employed,employed,student,student
location,europe,south_america,europe,NaN,north_america
annual_income,88160.0,72688.0,44697.0,NaN,24062.0
number_of_courses_viewed,3,3,3,3,4
interaction_count,4,7,4,6,5
lead_score,0.64,0.72,0.58,0.57,0.62
converted,1,1,1,0,1


In [156]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [157]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [158]:
df.nunique()

lead_source                    5
industry                       6
employment_status              4
location                       5
annual_income               4313
number_of_courses_viewed       7
interaction_count             14
lead_score                    99
converted                      2
dtype: int64

In [159]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

In [160]:
for category, fill_value in [(categorical, 'NA'), (numerical, 0)]:
    df[category] = df[category].fillna(fill_value)

df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

Question 1

In [161]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

Question 2

In [162]:
from itertools import combinations
for col1, col2 in combinations(numerical, 2):
    corr = df[col1].corr(df[col2])
    print(f"{col1} vs {col2}: {corr:.3f}")

annual_income vs number_of_courses_viewed: 0.161
annual_income vs interaction_count: 0.123
annual_income vs lead_score: 0.229
number_of_courses_viewed vs interaction_count: 0.722
number_of_courses_viewed vs lead_score: 0.757
interaction_count vs lead_score: 0.916


In [163]:
from sklearn.model_selection import train_test_split

In [164]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [165]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [166]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [167]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [168]:
df_train_converted = df_train['converted']
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [169]:
from sklearn.metrics import mutual_info_score

Question 3

In [173]:
def mutual_info_converted(series):
    return mutual_info_score(series, df_train_converted)

In [174]:
mi = df_train[categorical].apply(mutual_info_converted)
mi.sort_values(ascending=False)

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

Question 4

In [239]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [240]:
dv = DictVectorizer(sparse=False)

In [241]:
train_dicts = df_train[categorical].to_dict(orient='records')
val_dicts = df_val[categorical].to_dict(orient='records')

In [242]:
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

In [243]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

In [244]:
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The 

In [245]:
y_pred = model.predict_proba(X_val)[:, 1]

In [246]:
converted_decision = (y_pred >= 0.5)

In [248]:
original_score = (converted_decision == y_val).mean()
original_score

np.float64(0.655)

Question 5

In [252]:
features = df_train.columns.to_list()
features

['lead_source',
 'industry',
 'employment_status',
 'location',
 'annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [257]:
result = pd.DataFrame(columns=['feature', 'score', 'original_score', 'diff'])
for index, feature in enumerate(features):
    features_copy = features.copy()
    features_copy.remove(feature)

    dv = DictVectorizer(sparse=False)
    train_dicts = df_train[features_copy].to_dict(orient='records')
    X_train = dv.fit_transform(train_dicts)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    val_dicts = df_val[features_copy].to_dict(orient='records')
    X_val = dv.transform(val_dicts)

    y_pred = model.predict(X_val)

    converted_decision = (y_pred >= 0.5)
    score = (converted_decision == y_val).mean()

    result.loc[index] = [feature, score, original_score ,original_score - score]

In [266]:
result

,feature,score,original_score,diff
0,lead_source,0.642,0.655,0.013
1,industry,0.645,0.655,0.010
2,employment_status,0.645,0.655,0.010
3,location,0.645,0.655,0.010
4,annual_income,0.724,0.655,-0.069
5,number_of_courses_viewed,0.643,0.655,0.012
6,interaction_count,0.601,0.655,0.054
7,lead_score,0.644,0.655,0.011


In [287]:
subset_result = result.iloc[[0, 5, 6]] # Only take rows from possible answers of question 5
subset_result.iloc[result['diff'].abs().idxmin()]

feature           number_of_courses_viewed
score                                0.643
original_score                       0.655
diff                                 0.012
Name: 5, dtype: object

Question 6

In [297]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    dv = DictVectorizer(sparse=False)
    train_dicts = df_train.to_dict(orient='records')
    X_train = dv.fit_transform(train_dicts)

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    val_dicts = df_val.to_dict(orient='records')
    X_val = dv.transform(val_dicts)

    y_pred = model.predict(X_val)

    converted_decision = (y_pred >= 0.5)
    score = (converted_decision == y_val).mean()
    print(f"C: {C:.6f}".rstrip('0').rstrip('.') + f" Accuracy: {score:.3f}")

C: 0.000001 Accuracy: 0.598
C: 0.00001 Accuracy: 0.598
C: 0.0001 Accuracy: 0.613
C: 0.001 Accuracy: 0.645
